# The Last Mile Logistics Auditor
**Client:** Veridi Logistics | **Dataset:** Olist Brazilian E-Commerce

This notebook audits delivery performance by connecting logistics data (actual vs. estimated delivery dates) with customer sentiment (review scores) to identify where and why Veridi Logistics is failing its customers.

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## 1. Story 1 — The Schema Builder
1.Load the raw CSVs into the notebook

2.Join them into a single master dataset:

    -Join Reviews to Orders on order_id

    -Join Customers to Orders on customer_id

3.Check that you didn't accidentally duplicate rows

In [ ]:
orders   = pd.read_csv('data/olist_orders_dataset.csv')
reviews  = pd.read_csv('data/olist_order_reviews_dataset.csv')
customers = pd.read_csv('data/olist_customers_dataset.csv')
products  = pd.read_csv('data/olist_products_dataset.csv')
translation = pd.read_csv('data/product_category_name_translation.csv')

print('orders:   ', orders.shape)
print('reviews:  ', reviews.shape)
print('customers:', customers.shape)
print('products: ', products.shape)
print('translation: ', translation.shape)


In [ ]:
# Deduplicate reviews — keep one review per order to avoid row duplication on join
reviews_deduped = reviews.drop_duplicates(subset='order_id', keep='last')

# Join: orders <- reviews <- customers
df = orders.merge(reviews_deduped[['order_id', 'review_score']], on='order_id', how='left')
df = df.merge(customers[['customer_id', 'customer_state', 'customer_city']], on='customer_id', how='left')

print('Master dataset shape:', df.shape)
print('Duplicate order_ids:', df['order_id'].duplicated().sum())
df.head()


## 2. Story 2 — The "Real" Delay Calculator
Calculate delivery delay and classify orders.

In [ ]:
# Parse dates
date_cols = ['order_purchase_timestamp', 'order_estimated_delivery_date', 'order_delivered_customer_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

# Exclude undelivered orders
df_delivered = df[df['order_status'] == 'delivered'].copy()
df_excluded  = df[df['order_status'].isin(['canceled', 'unavailable'])].copy()

print('Delivered orders:', len(df_delivered))
print('Excluded orders (canceled/unavailable):', len(df_excluded))

In [ ]:
# Drop rows missing either date before calculating
df_delivered = df_delivered.dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date'])

# Days_Difference: positive = late, negative = early
df_delivered['days_difference'] = (
    df_delivered['order_delivered_customer_date'] - df_delivered['order_estimated_delivery_date']
).dt.days


def classify_delivery(days):
    if days > 5:
        return 'Super Late'
    elif days > 0:
        return 'Late'
    else:
        return 'On Time'

df_delivered['delivery_status'] = df_delivered['days_difference'].apply(classify_delivery)
df_delivered['delivery_status'].value_counts()

## 3. Story 3 — The Geographic Heatmap
Which states have the highest % of late deliveries?

In [ ]:
state_stats = df_delivered.groupby('customer_state').agg(
    total_orders=('order_id', 'count'),
    late_orders=('delivery_status', lambda x: (x != 'On Time').sum())
).reset_index()

state_stats['pct_late'] = (state_stats['late_orders'] / state_stats['total_orders'] * 100).round(2)
state_stats = state_stats.sort_values('pct_late', ascending=False)

plt.figure(figsize=(14, 6))
sns.barplot(data=state_stats, x='customer_state', y='pct_late', palette='Reds_r')
plt.title('% of Late Deliveries by State')
plt.xlabel('State')
plt.ylabel('% Late Orders')
plt.tight_layout()
plt.show()